In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from utils.modelling import *

In [ ]:
import os
if os.path.exists("../data/evaluation.parquet"):
    evaluation_df = pd.read_parquet("../data/evaluation.parquet")
else:
    evaluation_df = pd.DataFrame()

### VERSION 1

#### LOGISTIC REGRESSION

In [ ]:
insurance_claims = pd.read_parquet("../data/fraud_model_dataset_v1.parquet")

numeric_columns = insurance_claims.select_dtypes(include = ["int64", "int32", "float64"]).columns
categorical_columns = insurance_claims.select_dtypes(include = ["object"]).columns.drop("fraud_reported")
binary_map = {"fraud_reported": {"N": 0, "Y": 1}}

insurance_claims = encode_features(insurance_claims, categorical_columns, binary_map)
x_train, x_test, y_train, y_test = split_dataset(insurance_claims, "fraud_reported", 0.2, True)

preprocessor = ColumnTransformer(
    transformers = [
        ("numeric_scaled", StandardScaler(), numeric_columns)
    ], 
    remainder = "passthrough"
)

In [ ]:
lr = create_pipeline(LogisticRegression(), preprocessor)
lr.fit(x_train, y_train)
lr_results = classification_results(lr, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Logistic Regression", 
#                              "Initial", 0.5, lr_results, True)

In [ ]:
lr_ttc = optimise_threshold(lr, "balanced_accuracy", 5, x_train, y_train)
lr_ttc_results = classification_results(lr_ttc, x_train, x_test, y_train, y_test, True)

#evaluation_df = store_results(evaluation_df, "../data/evaluation.parquet", "V1", "Logistic Regression", 
#                              "Threshold", lr_ttc.best_threshold_, lr_ttc_results, True)